# 第12课：模块与日期时间

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用同目录的 [covid_sample.csv](covid_sample.csv)。课后独立练习见 [chapter12_模块与日期时间_课后练习.ipynb](chapter12_模块与日期时间_课后练习.ipynb)，数据仍是这份抽样，不要去读约 80 万行的全表。

本课回到数据分析主线：第9课洗字符串，本课洗日期。不要用手写切片硬拆年、月、日。

## 学习目标

1. 说明模块是别人写好的能力，并能认出第3课的 `import csv` 已经在用模块。
2. 比较三种 import 写法后，全班锁定 `from datetime import datetime`。
3. 区分日期字符串与 `datetime` 对象；用 `strptime` 按匹配的格式串把文本变成对象。
4. 用 `strftime` 把对象统一写成例如 `%Y/%m/%d` 的文本。
5. 比较两个日期的先后，相减后用 `.days` 得到跨度；空日期用 `if` 跳过。

## 学习知识点

| 模块 | 文本 ↔ 对象 | 比较与清洗 |
| --- | --- | --- |
| 模块是别人写好的工具包 | `datetime` 不是普通字符串 | 对象可以比较先后 |
| 三种 import，锁定一种 | `strptime` 文本 → 对象 | 相减得到时长 `.days` |
| 本班：`from datetime import datetime` | 格式串必须与数据一致 | 空串用 `if` 跳过 |
| `csv` 早就在用模块 | `strftime` 对象 → 统一文本 | 不讲 calendar / 时区 |

## 基础回顾与案例提问

[covid_sample.csv](covid_sample.csv) 是华盛顿州 2020 年上半年的县日记录。列：`date,county,state,fips,cases,deaths`。日期在文件里是文本，看起来像 `2020-01-21`。打开时使用 `encoding="utf-8-sig"`。

1. **R.1** 第3课写下 `import csv` 时，`csv` 是你自己定义的函数吗？模块是什么？
2. **R.2** `"2020-01-21"` 的 `type` 是什么？它和“真正的日期对象”有何不同？
3. **R.3** 若用格式 `"%d/%m/%Y"` 去解析 `"2020-01-21"`，会成功还是失败？先写预测。

**作答：** 预测：____；依据：____；验证后说明：____。

使用 Python 3；标准库 `csv` 与 `datetime`。从该文件夹启动内核。本课不讲 `calendar`、时区、`locale`，也不把 `try/except` 建成一套体系。不要使用 pandas。综合练习里的最早日期、最晚日期和跨度天数必须由你的代码算出，题面不提供这些验收数字。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


## 1. 模块：别人写好的能力

### 理论知识

**模块是一份已经写好的工具包。** 用 `import` 把它拿到当前程序里，就不必自己从零实现 CSV 解析或日期算法。

第3课开始每次读取表格都在做：

```python
import csv
```

`csv.reader` 不是你写的函数，是标准库提供的。本课要新请一位帮手：日期时间。原则相同——先导入，再调用。

模块不是“高阶选修”。会 `import csv` 就已经在用模块。

### 案例：先确认 csv 仍能读表头


In [1]:
import csv

with open("covid_sample.csv", "r", encoding="utf-8-sig", newline="") as file:
    rows = list(csv.reader(file))

header = rows[0]
body = rows[1:]
print("header:", header)
print("first date text:", body[0][0])
print("first county:", body[0][1])
print("type of date field:", type(body[0][0]))


header: ['date', 'county', 'state', 'fips', 'cases', 'deaths']
first date text: 2020-01-21
first county: Snohomish
type of date field: <class 'str'>


### 讲解

表头第一列应是 `date`。`encoding="utf-8-sig"` 用来去掉可能出现的 UTF-8 BOM，否则第一列名字可能带多余字符。

`body[0][0]` 看起来像日期，类型仍是 `str`。下一节才请 `datetime` 把它变成日期对象。本格只证明：模块 `csv` 你早就在用。

### 易错点与练习

1. **K1.1** 不写 `import csv` 能调用 `csv.reader` 吗？
2. **K1.2** 为什么本课读文件仍要用 `utf-8-sig`，而不是只写 `utf-8` 碰运气？

**作答：** 忘记 import：____；编码：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 三种 import，然后锁定一种

### 理论知识

日期工具在标准库模块 `datetime` 里，其中有一个**同名的类**也叫 `datetime`。三种常见写法都能用，但混着用会把全班绕晕。

1. `import datetime` → 调用 `datetime.datetime.strptime(...)`
2. `import datetime as dt` → 调用 `dt.datetime.strptime(...)`
3. `from datetime import datetime` → 调用 `datetime.strptime(...)`

**本班锁定第三种：** 从此格之后，只写

```python
from datetime import datetime
```

黑板上只留这一行。不要三种轮换着抄。

### 案例：三种写法解析同一天，然后锁定


In [2]:
import datetime as datetime_module

text = "2020-01-21"
fmt = "%Y-%m-%d"
print("style 1/2:", datetime_module.datetime.strptime(text, fmt))

from datetime import datetime

print("style 3 (锁定):", datetime.strptime(text, fmt))
print("locked import: from datetime import datetime")


style 1/2: 2020-01-21 00:00:00
style 3 (锁定): 2020-01-21 00:00:00
locked import: from datetime import datetime


### 讲解

第三种把**类** `datetime` 直接带进当前名字空间，所以写 `datetime.strptime`，不再写两层点号。

后面所有格子默认已经锁定这种写法。若你在自己的草稿里 `import datetime` 后又 `from datetime import datetime`，名字会被后一次覆盖——这正是全班要锁一种的原因。

### 易错点与练习

1. **K2.1** 锁定写法之后，`datetime.strptime` 里的 `datetime` 指模块还是类？
2. **K2.2** 若有人坚持 `import datetime`，他应该调用 `datetime.strptime` 还是 `datetime.datetime.strptime`？

**作答：** 锁定后的名字：____；第一种写法的调用：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. `datetime` 对象不是字符串

### 理论知识

**文件里读到的日期是文本；解析之后才是日期对象。** 文本可以拼接、切片；日期对象可以比较先后、相减得到时长。

看起来一样的 `"2020-01-21"` 仍然只是字符排成一列。不要用 `row[0][:4]` 当“标准做法”去取年份——那是碰巧 ISO 格式才容易切片。本课优先标准库。

### 案例：同一天的文本与对象


In [3]:
from datetime import datetime

text = "2020-01-21"
obj = datetime.strptime(text, "%Y-%m-%d")
print("text:", text, type(text))
print("obj:", obj, type(obj))
print("year:", obj.year, "month:", obj.month, "day:", obj.day)


text: 2020-01-21 <class 'str'>
obj: 2020-01-21 00:00:00 <class 'datetime.datetime'>
year: 2020 month: 1 day: 21


### 讲解

`strptime` 的结果类型是 `datetime.datetime`（打印 `type` 时看到的名字）。`obj.year` 是整数 2020，不是字符串 `"2020"`。

后面比较、相减都对 **对象** 做，不要对混杂格式的字符串做 `<`。

### 易错点与练习

1. **K3.1** `text == obj` 会是 True 吗？先预测再在练习格验证。
2. **K3.2** `obj.year` 的类型是什么？它能不能和整数 2020 直接比较？

**作答：** 文本等于对象？____；year 的类型：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. `strptime`：文本 → 对象

### 理论知识

**`strptime` = parse（解析）time。** 两个参数：日期文本，以及**描述这份文本长什么样**的格式串。

本课文件全是 `%Y-%m-%d`，即四位年、两位月、两位日，中间是短横线。

| 符号 | 含义 | 例子 |
| --- | --- | --- |
| `%Y` | 四位年 | 2020 |
| `%m` | 两位月 | 01 |
| `%d` | 两位日 | 21 |
| `-` 或 `/` | 必须与文本里的分隔符一致 | `-` |

格式串必须与数据一致，多一个空格、把 `-` 写成 `/`，都会失败。

### 案例：解析两天并看对象


In [4]:
from datetime import datetime

a = datetime.strptime("2020-01-21", "%Y-%m-%d")
b = datetime.strptime("2020-01-24", "%Y-%m-%d")
print(a)
print(b)
print(a.date())
print(b.date())


2020-01-21 00:00:00
2020-01-24 00:00:00
2020-01-21
2020-01-24


### 讲解

两条文本都按 `%Y-%m-%d` 解析。`a.date()` 只显示年-月-日，仍来自同一个对象。

Snohomish 县在抽样文件中的第一天正是 `2020-01-21`。这里只用两天字符串做语法，不代替你对全表的统计。

### 易错点与练习

1. **K4.1** 把格式写成 `"%Y/%m/%d"` 再去解析 `"2020-01-21"`，与数据哪里不一致？
2. **K4.2** 再解析 `"2020-06-30"`，打印它的 `month` 和 `day`。

**作答：** 斜线与短横：____；6 月 30 日的 month/day：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 格式串写错会失败（故障留在 Markdown）

### 理论知识

**格式串与数据不一致时，`strptime` 会抛出 `ValueError`。** 这不是电脑坏了，是你描述的样子和字符串对不上。

本课不把 `try/except` 讲成一套异常课。正确做法：看清文件里的样子，再选格式串。破坏性实验只写在 Markdown 里，不要放进最终 Run All 的代码格。

常见错法：文件是 ISO 的 `2020-01-21`，却写成欧洲习惯的 `%d/%m/%Y`。

### 案例：正确格式可以跑；错误格式只读说明


In [5]:
from datetime import datetime

ok = datetime.strptime("2020-01-21", "%Y-%m-%d")
print("matched format:", ok)
print("wrong format stays in Markdown, not in this cell")


matched format: 2020-01-21 00:00:00
wrong format stays in Markdown, not in this cell


### 讲解

下面这段是故障案例，**不要放进可运行单元格**：

```python
from datetime import datetime
datetime.strptime("2020-01-21", "%d/%m/%Y")
# ValueError: time data '2020-01-21' does not match format '%d/%m/%Y'
```

`%d/%m/%Y` 期待的是日/月/年且分隔符是斜线，例如 `21/01/2020`。给它 `2020-01-21`，对不上。

当堂验收时，写错格式串就对照这一行报错信息。

### 易错点与练习

1. **K5.1** 故障案例里，数据是哪一种样子、格式串又在等哪一种样子？
2. **K5.2** 若数据改成 `"21/01/2020"`，这时 `%d/%m/%Y` 还是 `%Y-%m-%d`？

**作答：** 不一致点：____；日/月/年文本该用：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. `strftime`：对象 → 统一文本

### 理论知识

**`strftime` = format（格式化）time。** 方向与 `strptime` 相反：已经有对象，要把它印成某一种文本。

同一天可以印成不同样子：

- `obj.strftime("%Y-%m-%d")` → `2020-01-21`
- `obj.strftime("%Y/%m/%d")` → `2020/01/21`

当堂验收要看到**统一格式后的若干行**。本课示例统一成 `%Y/%m/%d`。

### 案例：把 1 月 21 日和 24 日印成斜线格式


In [6]:
from datetime import datetime

a = datetime.strptime("2020-01-21", "%Y-%m-%d")
b = datetime.strptime("2020-01-24", "%Y-%m-%d")
print(a.strftime("%Y/%m/%d"))
print(b.strftime("%Y/%m/%d"))
print(a.strftime("%Y-%m-%d"))


2020/01/21
2020/01/24
2020-01-21


### 讲解

先 `strptime` 再 `strftime`：文本 → 对象 → 另一种文本。中间的对象才是可以比较、相减的东西。

不要对原始字符串做 `replace("-", "/")` 当作已经“掌握 datetime”。那只是改字符，没有得到日期对象。

### 易错点与练习

1. **K6.1** `strptime` 与 `strftime` 谁是“读进来”，谁是“印出去”？
2. **K6.2** 把 `a` 格式化成 `"%Y/%m/%d"`，手写你期望的字符串。

**作答：** 方向：____；期望文本：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 比较先后，相减得到天数

### 理论知识

**日期对象可以比较大小，也可以相减。** 后一天减去前一天，得到 `timedelta`；要跨度天数就读 `.days`。

```python
delta = later - earlier
print(delta.days)
```

`max(parsed)` / `min(parsed)` 也能用在已经解析好的对象列表上。全表的最早、最晚请在综合练习自己算，本格只用两天演示语法。

### 案例：2020-01-21 与 2020-01-24 相差几天


In [7]:
from datetime import datetime

a = datetime.strptime("2020-01-21", "%Y-%m-%d")
b = datetime.strptime("2020-01-24", "%Y-%m-%d")
delta = b - a
print("a < b:", a < b)
print("delta:", delta)
print("days:", delta.days)
print("a - b days:", (a - b).days)


a < b: True
delta: 3 days, 0:00:00
days: 3
a - b days: -3


### 讲解

24 日减 21 日，`.days` 为 3。反过来是 -3。比较 `a < b` 为 True，表示 21 日更早。

这两天的差不是全表跨度。全表要从文件里找出最早和最晚再相减，不要把 3 当成全表答案。

### 易错点与练习

1. **K7.1** `b - a` 的类型叫什么？天数在哪个属性里？
2. **K7.2** 若两天是同一天，`.days` 应是多少？

**作答：** timedelta 与 .days：____；同一天：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 空日期：用 `if` 跳过

### 理论知识

**本课抽样文件没有空日期，但仍要学会跳过空串。** 别的表常会出现 `""`。不要对空串调用 `strptime`。

口径：若 `text == ""`，则 `continue`（或记下来不进入列表）。本课不讲 `try/except` 体系；空与不空用 `if` 就能分。

真正读全表时，把这段判断放进循环即可。

### 案例：三天文本里有一个空串


In [8]:
from datetime import datetime

samples = ["2020-01-21", "", "2020-01-24"]
parsed = []
for text in samples:
    if text == "":
        print("skip empty")
        continue
    dt = datetime.strptime(text, "%Y-%m-%d")
    parsed.append(dt)
    print("parsed:", dt.strftime("%Y/%m/%d"))

print("kept:", len(parsed))


parsed: 2020/01/21
skip empty
parsed: 2020/01/24
kept: 2


### 讲解

三个输入留下两个对象。空串被跳过，循环继续后面的 `"2020-01-24"`。

若你在全表循环里忘记这个 `if`，一旦遇到空日期就会 `ValueError`。本文件碰巧没有空日期，不代表写法可以省掉判断。

### 易错点与练习

1. **K8.1** 空串应 `continue` 还是当成今天？本课口径是哪一种？
2. **K8.2** 对本课 `covid_sample.csv`，即使没有空日期，循环里仍写 `if text == ""` 有什么好处？

**作答：** 空串口径：____；仍然写 if 的理由：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 9. 循环解析本课文件，统一前 5 行

### 理论知识

**把整列 `date` 解析成对象，再统一印出文本。** 步骤与第3课相同：`open` → `csv.reader` → 分出 header/body → `for row in body`。

用 `header.index("date")` 找列，不要凭记忆写死下标 0 之后再也不核对。打开编码 `utf-8-sig`。

当堂验收之一：打印统一格式后的 5 行。下面演示可以印出前 5 个统一日期；全表最早、最晚和跨度留给 P1 你自己算。

### 案例：解析整列并打印 unified 的前 5 个


In [9]:
import csv
from datetime import datetime

with open("covid_sample.csv", "r", encoding="utf-8-sig", newline="") as file:
    rows = list(csv.reader(file))

header = rows[0]
body = rows[1:]
date_i = header.index("date")
print("date index:", date_i)

unified = []
for row in body:
    text = row[date_i]
    if text == "":
        continue
    dt = datetime.strptime(text, "%Y-%m-%d")
    unified.append(dt.strftime("%Y/%m/%d"))

print("unified[:5]:", unified[:5])


date index: 0
unified[:5]: ['2020/01/21', '2020/01/22', '2020/01/23', '2020/01/24', '2020/01/25']


### 讲解

输出里会出现抽样文件开头几天的斜线写法，例如第一行对应 Snohomish 的 `2020/01/21`。这只说明解析与格式化通了。

不要在这一格计算并公布全表跨度。P1 需要你用 `min` / `max`（或自己循环比较）得到最早、最晚，再相减取 `.days`。

### 易错点与练习

1. **K9.1** 为什么还要用 `header.index("date")`，而不是直接写 `row[0]` 再也不看表头？
2. **K9.2** `unified` 里装的是字符串还是 datetime 对象？若还要算跨度，应另外保留什么列表？

**作答：** 核对表头：____；跨度应对对象列表计算：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：统一格式 5 行 + 一个时间差

使用同一份 [covid_sample.csv](covid_sample.csv)。编码 `utf-8-sig`。锁定 `from datetime import datetime`。不要使用 pandas，不要读任何“全美国全表”文件。

空日期：若遇到空串就跳过。本文件可能没有空串，判断仍要写。

最早日期、最晚日期、跨度天数必须由代码打印，不要把数字写死在程序里，也不要抄题面不存在的答案。

### P1.1　读取并打印 5 行统一格式

读取文件，解析 `date` 列，用 `strftime` 统一成 `%Y/%m/%d`，打印前 5 个。同时打印 `header` 与 `date` 下标。


In [ ]:
# P1.1: Read covid_sample.csv, parse dates, print unified[:5].


### P1.2　跳过空串，保留解析后的对象列表

循环里对空串 `continue`。把成功解析的 **datetime 对象** 放进列表（不要只留格式化后的字符串）。打印保留了多少个对象。

**作答：** 空串规则：____；对象个数如何得到：____。


In [ ]:
# P1.2: Skip empty strings; keep a list of datetime objects.


### P1.3　最早、最晚与跨度天数

对对象列表求最早、最晚，相减并打印 `.days`。再写一句：这个跨度的单位是天，还是秒？

**作答：**

1. 最早：____
2. 最晚：____
3. 跨度天数（代码打印值）：____


In [ ]:
# P1.3: Compute earliest, latest, and span in days from parsed objects.


## 本章总结

1. 模块是别人写好的工具包；`csv` 从第3课起就在用。
2. 三种 import 看过之后，本班只锁 `from datetime import datetime`。
3. 文件里的日期是字符串；`strptime` 变成对象，`strftime` 再变成统一文本。
4. 格式串必须与数据一致；写错会 `ValueError`（故障代码留在 Markdown）。
5. 对象可比较、可相减；`.days` 是跨度天数。空串用 `if` 跳过。

课后请打开 [chapter12_模块与日期时间_课后练习.ipynb](chapter12_模块与日期时间_课后练习.ipynb)，仍使用 [covid_sample.csv](covid_sample.csv) 独立完成。P1 统计 2020 年 6 月的行数，P2 解释错误格式串，P3 选做某一县的首末跨度。
